In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tf_keras.models import Model
from tf_keras.layers import (Input, LSTM, Dense, Dropout,
                              BatchNormalization, Bidirectional)
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint






In [19]:
def add_technical_indicators(df):
    df = df.copy()
    df['SMA_5']     = df['Close'].rolling(5).mean()
    df['SMA_20']    = df['Close'].rolling(20).mean()
    ema12           = df['Close'].ewm(span=12, adjust=False).mean()
    ema26           = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']      = ema12 - ema26
    df['MACD_sig']  = df['MACD'].ewm(span=9, adjust=False).mean()
    delta           = df['Close'].diff()
    gain            = delta.clip(lower=0).rolling(14).mean()
    loss            = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI']       = 100 - (100 / (1 + gain / (loss + 1e-9)))
    sma20           = df['Close'].rolling(20).mean()
    std20           = df['Close'].rolling(20).std()
    df['BB_width']  = (4 * std20) / (sma20 + 1e-9)
    hl              = df['High'] - df['Low']
    hcp             = (df['High'] - df['Close'].shift()).abs()
    lcp             = (df['Low']  - df['Close'].shift()).abs()
    df['ATR']       = pd.concat([hl, hcp, lcp], axis=1).max(axis=1).rolling(14).mean()
    df['Vol_ratio'] = df['Volume'] / (df['Volume'].rolling(20).mean() + 1e-9)
    df['Return_1d'] = df['Close'].pct_change(1)
    df['Return_5d'] = df['Close'].pct_change(5)
    df['LogReturn'] = np.log(df['Open'] / df['Open'].shift(1))
    return df.dropna()
TARGET_COL = 'LogReturn'   # what we predict
PRICE_COL  = 'Open' 
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA',
    'AVGO', 'AMD', 'INTC', 'QCOM', 'TXN', 'MU', 'AMAT', 'LRCX',
    'ORCL', 'ADBE', 'CRM', 'INTU', 'SNPS', 'CDNS', 'ANSS',
    'NOW', 'WDAY', 'TEAM', 'ZS', 'PANW', 'CRWD',
    'CSCO', 'IBM', 'HPQ', 'DELL', 'WDC', 'STX', 'ANET', 'FFIV'
]
WINDOW     = 60
FEATURES   = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA_5', 'SMA_20', 'MACD', 'MACD_sig',
    'RSI', 'BB_width', 'ATR', 'Vol_ratio',
    'Return_1d', 'Return_5d', 'LogReturn'
]
TARGET_IDX = FEATURES.index('LogReturn')

SAVE_DIR   = '/content/drive/MyDrive/DL4AI/models/'
os.makedirs(SAVE_DIR, exist_ok=True)

def split_stock(df, train=0.7, val=0.15):
    n = len(df)
    t = int(n * train)
    v = int(n * (train + val))
    return df.iloc[:t], df.iloc[t:v], df.iloc[v:]

def split_and_scale(train, val, test):
    scaler  = RobustScaler()
    train_s = scaler.fit_transform(train[FEATURES])
    val_s   = scaler.transform(val[FEATURES])
    test_s  = scaler.transform(test[FEATURES])
    return train_s, val_s, test_s, scaler

def create_windows(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window, TARGET_IDX])
    return np.array(X), np.array(y)

def create_windows_with_context(prev_tail, curr_data, window):
    combined = np.concatenate([prev_tail[-window:], curr_data], axis=0)
    X, y = [], []
    for i in range(len(curr_data)):
        X.append(combined[i:i+window])
        y.append(combined[i+window, TARGET_IDX])
    return np.array(X), np.array(y)

def inverse_target(y_returns_scaled, scaler, df_prices, start_offset):
    """
    1. Unscale the predicted log returns
    2. Reconstruct prices using the actual preceding Open price
    """
    dummy = np.zeros((len(y_returns_scaled), len(FEATURES)))
    dummy[:, TARGET_IDX] = y_returns_scaled.flatten()
    log_returns = scaler.inverse_transform(dummy)[:, TARGET_IDX]

    # Anchor on the actual Open price just before each prediction window
    prices = df_prices['Open'].values
    reconstructed = []
    for i, lr in enumerate(log_returns):
        anchor_idx = start_offset + i + WINDOW - 1   # last known Open
        anchor_price = prices[min(anchor_idx, len(prices)-1)]
        reconstructed.append(anchor_price * np.exp(lr))

    return np.array(reconstructed)


def predict_next_day(model, scaler, df_full):
    scaled   = scaler.transform(df_full[FEATURES])
    X_input  = scaled[-WINDOW:].reshape(1, WINDOW, len(FEATURES))
    pred_s   = model.predict(X_input, verbose=0)

    # Unscale the predicted log return
    dummy    = np.zeros((1, len(FEATURES)))
    dummy[0, TARGET_IDX] = pred_s[0, 0]
    log_ret  = scaler.inverse_transform(dummy)[0, TARGET_IDX]

    # Reconstruct price: anchor on last known Open
    last_open = df_full['Open'].values[-1]
    return last_open * np.exp(log_ret)


def build_lstm(input_shape):
    from tf_keras.optimizers import Adam
    from tf_keras.regularizers import l2

    inp = Input(shape=input_shape)

    # Lighter architecture for ~1000 samples
    x = LSTM(64, return_sequences=True,
             kernel_regularizer=l2(1e-4))(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)                    # higher dropout to fight overfitting

    x = LSTM(32, return_sequences=False,
             kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(16, activation='relu')(x)
    x = Dropout(0.2)(x)
    out = Dense(1)(x)                      # predicts log return

    model = Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
        loss='huber'
    )
    return model


In [20]:
MIN_ROWS     = WINDOW + 50
raw_data     = {}
valid_stocks = {}   # ← this is what was missing

for ticker in TICKERS:
    df = yf.download(ticker, start='2015-01-01', end='2026-02-28',
                     auto_adjust=True, progress=False)
    df = df[['Open','High','Low','Close','Volume']]
    df.columns = ['Open','High','Low','Close','Volume']
    df = df.apply(pd.to_numeric, errors='coerce').dropna()
    df = add_technical_indicators(df)          # ← must be before split
    raw_data[ticker] = df

    train, val, test = split_stock(df)
    if min(len(train), len(val), len(test)) < MIN_ROWS:
        print(f"{ticker} skipped: too short")
        continue
    valid_stocks[ticker] = {'train': train, 'val': val, 'test': test, 'full': df}
    print(f"{ticker} | rows: {len(df)} | splits: {len(train)}/{len(val)}/{len(test)}")

print(f"\nValid stocks: {len(valid_stocks)}")


AAPL | rows: 2786 | splits: 1950/418/418
MSFT | rows: 2786 | splits: 1950/418/418
GOOGL | rows: 2786 | splits: 1950/418/418
AMZN | rows: 2786 | splits: 1950/418/418
META | rows: 2786 | splits: 1950/418/418
NVDA | rows: 2786 | splits: 1950/418/418
TSLA | rows: 2786 | splits: 1950/418/418
AVGO | rows: 2786 | splits: 1950/418/418
AMD | rows: 2786 | splits: 1950/418/418
INTC | rows: 2786 | splits: 1950/418/418
QCOM | rows: 2786 | splits: 1950/418/418
TXN | rows: 2786 | splits: 1950/418/418
MU | rows: 2786 | splits: 1950/418/418
AMAT | rows: 2786 | splits: 1950/418/418
LRCX | rows: 2786 | splits: 1950/418/418
ORCL | rows: 2786 | splits: 1950/418/418
ADBE | rows: 2786 | splits: 1950/418/418
CRM | rows: 2786 | splits: 1950/418/418
INTU | rows: 2786 | splits: 1950/418/418
SNPS | rows: 2786 | splits: 1950/418/418
CDNS | rows: 2786 | splits: 1950/418/418


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ANSS']: YFTzMissingError('possibly delisted; no timezone found')


ANSS skipped: too short
NOW | rows: 2786 | splits: 1950/418/418
WDAY | rows: 2786 | splits: 1950/418/418
TEAM | rows: 2550 | splits: 1785/382/383
ZS | rows: 1980 | splits: 1386/297/297
PANW | rows: 2786 | splits: 1950/418/418
CRWD | rows: 1669 | splits: 1168/250/251
CSCO | rows: 2786 | splits: 1950/418/418
IBM | rows: 2786 | splits: 1950/418/418
HPQ | rows: 2786 | splits: 1950/418/418
DELL | rows: 2377 | splits: 1663/357/357
WDC | rows: 2786 | splits: 1950/418/418
STX | rows: 2786 | splits: 1950/418/418
ANET | rows: 2786 | splits: 1950/418/418
FFIV | rows: 2786 | splits: 1950/418/418

Valid stocks: 35


In [21]:
def train_ticker(ticker):
    s = valid_stocks[ticker]
    train_s, val_s, test_s, scaler = split_and_scale(s['train'], s['val'], s['test'])

    X_tr, y_tr = create_windows(train_s, WINDOW)
    X_v,  y_v  = create_windows_with_context(train_s, val_s,  WINDOW)
    X_te, y_te = create_windows_with_context(val_s,   test_s, WINDOW)

    model = build_lstm((WINDOW, len(FEATURES)))
    callbacks = [
        EarlyStopping(patience=20, restore_best_weights=True, min_delta=1e-6),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7),
        ModelCheckpoint(f"{SAVE_DIR}{ticker}_best.keras",
                        save_best_only=True, monitor='val_loss', verbose=0)
    ]
    history = model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
                        epochs=200, batch_size=32, callbacks=callbacks, verbose=1)

    y_pred = model.predict(X_te, verbose=0)

    # ── Reconstruct real prices from log returns ──────────────────────
    n_train = len(s['train'])
    n_val   = len(s['val'])
    full_df = s['full']

    y_te_inv   = inverse_target(y_te,   scaler, full_df, n_train + n_val)
    y_pred_inv = inverse_target(y_pred, scaler, full_df, n_train + n_val)

    mae      = mean_absolute_error(y_te_inv, y_pred_inv)
    rmse     = np.sqrt(mean_squared_error(y_te_inv, y_pred_inv))
    next_day = predict_next_day(model, scaler, full_df)

    print(f"\n{ticker} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | Next day Open: ${next_day:.2f}")
    model.save(f"{SAVE_DIR}{ticker}_lstm.keras")
    model.save(f"{SAVE_DIR}{ticker}_lstm.keras")
    joblib.dump(scaler, f"{SAVE_DIR}{ticker}_scaler.pkl")  # ← add this line


    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history.history['loss'],     label='Train Loss', color='royalblue')
    axes[0].plot(history.history['val_loss'], label='Val Loss',   color='tomato')
    axes[0].set_title(f'{ticker} — Training & Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Huber Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(y_te_inv,   label='Actual',    color='royalblue')
    axes[1].plot(y_pred_inv, label='Predicted', color='orange')
    axes[1].set_title(f'{ticker} — Next-Day Open Price Prediction')
    axes[1].set_xlabel('Trading Days (Test Set)')
    axes[1].set_ylabel('Open Price (USD)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'{ticker}  |  MAE: {mae:.4f}  |  RMSE: {rmse:.4f}', fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}{ticker}_prediction.png", dpi=150)
    plt.show()

    return model, scaler, y_te_inv, y_pred_inv, mae, rmse


In [ ]:
import joblib

TICKERS = [
    # Big Tech
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA',
    # Semiconductors
    'AVGO', 'AMD', 'INTC', 'QCOM', 'TXN', 'MU', 'AMAT', 'LRCX',
    # Software
    'ORCL', 'ADBE', 'CRM', 'INTU', 'SNPS', 'CDNS', 'ANSS',
    # Cloud & Services
    'NOW', 'WDAY', 'TEAM', 'ZS', 'PANW', 'CRWD',
    # Others
    'CSCO', 'IBM', 'HPQ', 'DELL', 'WDC', 'STX', 'ANET', 'FFIV'
]

results = {}

for ticker in TICKERS:
    model_path  = f"{SAVE_DIR}{ticker}_best.keras"
    scaler_path = f"{SAVE_DIR}{ticker}_scaler.pkl"

    # ── Skip if already trained ────────────────────────────────────────
    if os.path.exists(model_path) and os.path.exists(scaler_path):
        print(f" {ticker} already trained — skipping")
        continue
    # ──────────────────────────────────────────────────────────────────

    print(f"\n{'='*40}\nTraining: {ticker}\n{'='*40}")
    try:
        model, scaler, y_te, y_pred, mae, rmse = train_ticker(ticker)
        results[ticker] = {'mae': mae, 'rmse': rmse}
    except Exception as e:
        print(f" {ticker} FAILED: {e}")
        results[ticker] = {'mae': None, 'rmse': None}

# Save summary
results_df = pd.DataFrame(results).T.sort_values('mae')
results_df.to_csv(f"{SAVE_DIR}all_results.csv")
print(results_df)


 AAPL already trained — skipping
 MSFT already trained — skipping
 GOOGL already trained — skipping
 AMZN already trained — skipping

Training: META
Epoch 1/200
60/60 [==============================] - 12s 92ms/step - loss: 0.6510 - val_loss: 0.4973 - lr: 5.0000e-04
Epoch 2/200
60/60 [==============================] - 4s 67ms/step - loss: 0.5305 - val_loss: 0.4993 - lr: 5.0000e-04
Epoch 3/200
60/60 [==============================] - 5s 86ms/step - loss: 0.4896 - val_loss: 0.5055 - lr: 5.0000e-04
Epoch 4/200
60/60 [==============================] - 4s 66ms/step - loss: 0.4712 - val_loss: 0.5045 - lr: 5.0000e-04
Epoch 5/200
60/60 [==============================] - 4s 65ms/step - loss: 0.4582 - val_loss: 0.5028 - lr: 5.0000e-04
Epoch 6/200
60/60 [==============================] - 5s 88ms/step - loss: 0.4352 - val_loss: 0.5030 - lr: 5.0000e-04
Epoch 7/200
60/60 [==============================] - 4s 66ms/step - loss: 0.4361 - val_loss: 0.4986 - lr: 5.0000e-04
Epoch 8/200
60/60 [============